In [2]:
# importing liabraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer # Added SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# loading the data set
data1=pd.read_csv("client_train.csv")
data2=pd.read_csv("invoice_train.csv")

data3=pd.read_csv("client_test.csv")
data4=pd.read_csv("invoice_test.csv")


In [ ]:
# merging datasets
df1=pd.merge(data1,data2,on='client_id',how='left')
df2=pd.merge(data3,data4,on='client_id',how='left')

# DATA CLEANING AND EDA

In [ ]:
# train data set
df1.head()

In [ ]:
df1.describe()

In [ ]:
df1.info()

In [ ]:
# duplicated values
df1.duplicated().sum()

In [ ]:
df1.drop_duplicates(inplace=True)

In [ ]:
# null values
df1.isnull().sum()/len(df1)*100

In [ ]:
numerical_col=df1.select_dtypes(int,float).columns
categorical_col=df1.select_dtypes(object).columns
numerical_col,categorical_col

In [ ]:
# Convert dates
df1['creation_date'] = pd.to_datetime(df1['creation_date'])
df1['invoice_date'] = pd.to_datetime(df1['invoice_date'])

# Extract date features
df1['creation_year'] = df1['creation_date'].dt.year
df1['creation_month'] = df1['creation_date'].dt.month

df1['invoice_year'] = df1['invoice_date'].dt.year
df1['invoice_month'] = df1['invoice_date'].dt.month
df1['invoice_dayofweek'] = df1['invoice_date'].dt.dayofweek

In [ ]:
# feature engeneering
df1['total_consumption'] = (
    df1['consommation_level_1'] +
    df1['consommation_level_2'] +
    df1['consommation_level_3'] +
    df1['consommation_level_4']
)


df1['level1_ratio'] = (
    df1['consommation_level_1'] /
    df1['total_consumption']
)

df1['index_difference'] = (
    df1['new_index'] -
    df1['old_index']
)
df1['avg_monthly_consumption'] = (
    df1['total_consumption'] /
    df1['months_number']
)
df1['consumption_per_counter'] = (
    df1['total_consumption'] /
    df1['counter_coefficient']
)
df1['invoice_date'] = pd.to_datetime(df1['invoice_date'])

df1['invoice_year_month'] = (
    df1['invoice_date'].dt.to_period('M')
)

# VISUALIZATION

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df1['total_consumption'], bins=50, kde=True)
plt.title('Distribution of Total Consumption')
plt.show()

In [ ]:
consumption_cols = [
    'consommation_level_1',
    'consommation_level_2',
    'consommation_level_3',
    'consommation_level_4'
]

df1[consumption_cols].mean().plot(
    kind='bar',
    figsize=(8,5)
)

plt.title('Average Consumption by Level')
plt.ylabel('Consumption')
plt.show()

In [ ]:
monthly = df1.groupby(
    df1['invoice_date'].dt.to_period('M')
)['total_consumption'].mean()

monthly.plot(figsize=(12,5))
plt.title('Monthly Average Consumption')
plt.ylabel('Consumption')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    x='counter_type',
    y='total_consumption',
    data=df1
)

plt.xticks(rotation=45)
plt.title('Consumption by Counter Type')
plt.show()

In [ ]:
region_consumption = (
    df1.groupby('region')['total_consumption']
      .mean()
      .sort_values(ascending=False)
      .head(10)
)

plt.figure(figsize=(12,6))
region_consumption.plot(kind='bar')
plt.title('Top 10 Regions by Average Consumption')
plt.ylabel('Average Consumption')
plt.show()

In [ ]:
numeric_cols = [
    'counter_number',
    'counter_code',
    'counter_coefficient',
    'consommation_level_1',
    'consommation_level_2',
    'consommation_level_3',
    'consommation_level_4',
    'old_index',
    'new_index',
    'months_number',
    'total_consumption',
    'index_difference'
]

plt.figure(figsize=(14,10))

sns.heatmap(
    df1[numeric_cols].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Feature Correlation Matrix')
plt.show()

MODELLING


In [ ]:
# feature and target selection
x=df1.drop(['client_id','creation_date','invoice_date','invoice_year_month','target'],axis=1)
y=df1['target']


In [ ]:
x = pd.get_dummies(x,drop_first=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
# preprocessing

# Handle potential infinite values by replacing them with NaN
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Impute NaN values using SimpleImputer
imputer = SimpleImputer(strategy='mean')

# Fit the imputer on X_train and transform both X_train and X_test
X_train = imputer.fit_transform(X_train) # X_train becomes a NumPy array here
X_test = imputer.transform(X_test) # X_test becomes a NumPy array here

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

LOGISTIC CLASSIFICATION

In [ ]:
# base model
lr=LogisticRegression()
lr.fit(X_train,y_train)
y_pred=lr.predict(X_test)
print(classification_report(y_test,y_pred))

SVM CLASSIFICATION

DECISON TREE CLASSIFICATION

In [ ]:
dt=DecisionTreeClassifier(max_depth=5)
dt.fit(X_train,y_train)
y_pred=dt.predict(X_test)
print(classification_report(y_test,y_pred))

RANDOM FOREST CLASSIFICATION

In [ ]:
rf=RandomForestClassifier(max_depth=5,n_estimators=100)
rf.fit(X_train,y_train)
y_pred=rf.predict(X_test)
print(classification_report(y_test,y_pred))

XGBOOST

In [ ]:
xg=XGBClassifier()
xg.fit(X_train,y_train)
y_pred=xg.predict(X_test)
print(classification_report(y_test,y_pred))


MODELs EVALUATION

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = [
    ('Logistic Regression', LogisticRegression(max_iter=1000)),
    ('Decision Tree', DecisionTreeClassifier(random_state=42, max_depth=10)),
    ('Random Forest', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_depth=10
    )),
    ('XGBoost', XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        n_estimators=100,
        max_depth=6,
        n_jobs=-1,
        verbosity=0
    ))
]

# Store evaluation metrics
results = []

for model_name, model in models:
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1-score': f1_score(y_test, y_pred, average='weighted')
    })

# Create DataFrame
results_df = pd.DataFrame(results)

print("\nModel Performance Comparison:")
print(results_df.round(4))

# Plot
ax = results_df.set_index('Model').plot(
    kind='bar',
    figsize=(12, 6),
    width=0.8,
    colormap='viridis'
)

plt.title('Model Performance Comparison')
plt.xlabel('Models')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=3)

plt.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# TUNNING
the best performing model

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV # Added this import

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

xgb = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

In [ ]:
grid.fit(X_train, y_train)

feature importance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import plot_importance

# Get the best tuned model
best_model = grid.best_estimator_

# Get feature names
if hasattr(X_train, "columns"):
    feature_names = X_train.columns
else:
    feature_names = [f"Feature_{i}" for i in range(X_train.shape[1])]

# Get feature importances
importance = best_model.feature_importances_

# Create DataFrame
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

# Sort descending
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

# Display top 15 features
print(feature_importance.head(15))

# Plot
plt.figure(figsize=(10,8))

plt.barh(
    feature_importance["Feature"][:15][::-1],
    feature_importance["Importance"][:15][::-1]
)

plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Top 15 Important Features (Tuned XGBoost)")

# Add labels
for index, value in enumerate(feature_importance["Importance"][:15][::-1]):
    plt.text(value, index, f"{value:.3f}", va="center")

plt.tight_layout()
plt.show()

TESTING

In [ ]:
data1=pd.read_csv("client_test.csv")
data2=pd.read_csv("invoice_test.csv")
data=pd.merge(data1,data2,on='client_id',how='left')
data